# Section 6. 긴 문서를 검색 가능한 단위로 만들기

공개 실습용 notebook입니다. Jupyter에서 위에서 아래로 실행합니다. API를 호출하지 않습니다.

**API key:** 이 Section의 기본 실습에는 필요하지 않습니다.

In [ ]:
from pathlib import Path
from statistics import mean

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def find_corpus() -> Path:
    candidates = [
        Path("../data/section06_onboarding.txt"),
        Path(__file__).resolve().parents[1] / "data" / "section06_onboarding.txt"
        if "__file__" in globals()
        else Path("__missing__"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("section06_onboarding.txt를 찾지 못했습니다.")


corpus_path = find_corpus()
source_text = corpus_path.read_text(encoding="utf-8")
print("corpus:", corpus_path)
print("characters:", len(source_text))

## 원본 Document 만들기

In [ ]:
document = Document(
    page_content=source_text,
    metadata={
        "source_id": "section06-onboarding",
        "title": "가상 AMSL 프로젝트 온보딩 안내",
        "source_type": "txt",
    },
)
print(document.metadata)

## 두 가지 설정으로 분할하기

In [ ]:
SETTINGS = {
    "small": {"chunk_size": 180, "chunk_overlap": 20},
    "large": {"chunk_size": 500, "chunk_overlap": 80},
}


def split_with_metadata(source: Document, label: str, options: dict) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        **options,
        add_start_index=True,
        keep_separator="end",
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents([source])
    for index, chunk in enumerate(chunks):
        start = chunk.metadata["start_index"]
        chunk.metadata.update(
            {
                "chunk_id": f"{source.metadata['source_id']}:{label}:{index:03d}",
                "char_start": start,
                "char_end": start + len(chunk.page_content),
                "split_setting": label,
            }
        )
    return chunks


results = {
    label: split_with_metadata(document, label, options)
    for label, options in SETTINGS.items()
}

## 길이와 경계 관찰하기

In [ ]:
for label, chunks in results.items():
    lengths = [len(chunk.page_content) for chunk in chunks]
    print(f"\n[{label}] chunks={len(chunks)} min={min(lengths)} mean={mean(lengths):.1f} max={max(lengths)}")
    for chunk in chunks:
        preview = chunk.page_content[:80].replace("\n", " ↵ ")
        print(chunk.metadata["chunk_id"], chunk.metadata["char_start"], repr(preview))

## metadata와 원문 위치 검증하기

공백 정규화가 없는 현재 corpus에서는 `char_start:char_end`로 원문을 다시 잘랐을 때
`page_content`와 일치해야 합니다.

In [ ]:
for label, chunks in results.items():
    chunk_ids = set()
    for chunk in chunks:
        metadata = chunk.metadata
        assert metadata["source_id"] == "section06-onboarding"
        assert metadata["chunk_id"] not in chunk_ids
        chunk_ids.add(metadata["chunk_id"])

        restored = source_text[metadata["char_start"] : metadata["char_end"]]
        assert restored == chunk.page_content, (
            f"원문 위치 불일치: {metadata['chunk_id']}"
        )
    print(label, "metadata/location validation: PASS")

## 일부러 metadata를 잃어버리는 실패 예시

In [ ]:
broken_chunks = [Document(page_content=chunk.page_content) for chunk in results["small"]]
missing_source_count = sum("source_id" not in chunk.metadata for chunk in broken_chunks)
print("source_id가 사라진 chunk:", missing_source_count, "/", len(broken_chunks))
assert missing_source_count == len(broken_chunks)

## 정리 질문

1. 두 설정에서 chunk 수와 평균 길이는 어떻게 달랐나요?
2. 문맥이 끊긴 경계를 하나 찾으세요.
3. overlap으로 내용이 반복된 사례를 하나 찾으세요.
4. Section 7의 검색 실험에서 어느 설정이 더 유리할지 가설을 적으세요.